# 02 · Недельный прогноз всего магазина CA_1

**3 049 товаров**, три категории, прогноз на следующую неделю. Сравниваем простые
методы и CatBoost, добавляем 27 признаков, проверяем четыре временных окна и Optuna.

| Метод на 25 апреля — 22 мая 2016 | MAE | WAPE | Bias |
|---|---:|---:|---:|
| Среднее за 4 недели | 3.8097 | 34.67% | −3.65% |
| CatBoost после подбора | **3.5512** | **32.31%** | −5.84% |

MAE снизился на **6.79%**. Это недельный эксперимент одного магазина;
далее исследование переходит к дневному M5 в [ноутбуке 03](03_kaggle_wrmsse.ipynb).

<a id="environment"></a>

## 0. Окружение

Используем Python из `.venv` проекта. Загрузка данных, базовые признаки и метрики вынесены в `src`. Дополнительные признаки и эксперименты с моделями пока находятся в этом ноутбуке.

Первая ячейка проверяет окружение, находит папку проекта и задаёт пути к данным и результатам.

In [1]:
from pathlib import Path
from time import perf_counter
import json
import sys

import pandas as pd

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "requirements.txt").is_file() and (path / "src").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Откройте ноутбук из папки demand-planner или notebooks.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Выберите Python из окружения .venv проекта.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import download_m5, load_subset, build_weekly_table
from src.features import (
    BASELINE_METHODS,
    FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    add_weekly_features,
    split_by_time,
)
from src.backtest import evaluate_forecasts

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "results"
STORE_ID = "CA_1"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)

print(f"Python: {sys.version.split()[0]}")
print(f"Проект: {PROJECT_ROOT}")
print(f"Магазин: {STORE_ID}")

Python: 3.12.14
Проект: /Users/daniilsestov/Desktop/pet_projects/demand-planner
Магазин: CA_1


<a id="data"></a>

## 1. Все товары магазина

Берём все 3 049 товаров CA_1 из M5. `category=None` оставляет все категории, `n_items=None` — все товары магазина. Случайной выборки здесь нет.

`sales_train_validation.csv` содержит историю до 24 апреля 2016 года. Дополнительные четыре недели из `sales_train_evaluation.csv` подключим только после выбора модели в разделе 10.

In [2]:
display(download_m5(DATA_RAW))

started = perf_counter()
calendar, sales, prices, source_shapes = load_subset(
    DATA_RAW,
    store_id=STORE_ID,
    category=None,
    n_items=None,
    seed=RANDOM_STATE,
)
load_seconds = perf_counter() - started

assert sales["store_id"].eq(STORE_ID).all()
category_items = sales.groupby("cat_id").agg(items=("item_id", "nunique"))
day_columns = [column for column in sales.columns if column.startswith("d_")]

display(category_items)
print(f"Всего товаров: {len(sales):,}")
print(f"Дней истории: {len(day_columns):,}")
print(f"Чтение данных: {load_seconds:.2f} сек.")

,file,bytes,md5
0,calendar.csv,103469,3ffeab2991b0c8e861d008b39ea4c95c
1,sales_train_validation.csv,120007726,26a366a25beb57b0a8f4c7b148758f94
2,sell_prices.csv,203395785,08c591caa99e55daf3e0ccac913f7c85


,items
cat_id,
FOODS,1437
HOBBIES,565
HOUSEHOLD,1047


Всего товаров: 3,049
Дней истории: 1,913
Чтение данных: 3.30 сек.


В таблице продаж одна строка соответствует товару в магазине, а дни находятся в отдельных столбцах. Календарь связывает номера дней с датами, событиями и SNAP. Цены загружаем только для товаров CA_1.

Ниже проверяем размеры таблиц и несколько идентификаторов.

In [3]:
loaded_tables = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(frame),
            "columns": frame.shape[1],
            "memory_mb": frame.memory_usage(deep=True).sum() / 1024**2,
        }
        for name, frame in [("calendar", calendar), ("sales", sales), ("prices", prices)]
    ]
)

display(loaded_tables.round(2))
display(sales[["item_id", "dept_id", "cat_id", "store_id"]].head())

,table,rows,columns,memory_mb
0,calendar,1969,14,0.24
1,sales,3049,1919,22.58
2,prices,698412,4,29.82


,item_id,dept_id,cat_id,store_id
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1


<a id="weekly"></a>

## 2. Недельные продажи

Полная дневная таблица этого магазина содержала бы почти 5,8 млн строк. Для недельной задачи складываем дни прямо из широкой таблицы. Получаем 273 полные недели и 832 377 строк «товар — неделя».

Неделя идёт с понедельника по воскресенье. Начальные неполные недели исключаем, нулевые продажи сохраняем. `first_price_date` — первая дата с записью о цене в доступной истории. Запись о цене сама по себе ничего не говорит об остатках товара.

In [4]:
started = perf_counter()
weekly = build_weekly_table(calendar, sales, prices)
weekly_seconds = perf_counter() - started

assert weekly["item_id"].nunique() == sales["item_id"].nunique()
assert len(weekly) == len(sales) * weekly["week_end"].nunique()

display(weekly[["week_end", "item_id", "cat_id", "weekly_sales", "first_price_date"]].head(10))
print(f"Полных недель: {weekly['week_end'].nunique()}")
print(f"Строк товар-неделя: {len(weekly):,}")
print(f"Память недельной таблицы: {weekly.memory_usage(deep=True).sum() / 1024**2:.2f} МБ")
print(f"Подготовка недель: {weekly_seconds:.2f} сек.")

,week_end,item_id,cat_id,weekly_sales,first_price_date
0,2011-02-06,FOODS_1_001,FOODS,9,2011-01-29
1,2011-02-13,FOODS_1_001,FOODS,7,2011-01-29
2,2011-02-20,FOODS_1_001,FOODS,10,2011-01-29
3,2011-02-27,FOODS_1_001,FOODS,14,2011-01-29
4,2011-03-06,FOODS_1_001,FOODS,16,2011-01-29
5,2011-03-13,FOODS_1_001,FOODS,12,2011-01-29
6,2011-03-20,FOODS_1_001,FOODS,3,2011-01-29
7,2011-03-27,FOODS_1_001,FOODS,8,2011-01-29
8,2011-04-03,FOODS_1_001,FOODS,1,2011-01-29
9,2011-04-10,FOODS_1_001,FOODS,8,2011-01-29


Полных недель: 273
Строк товар-неделя: 832,377
Память недельной таблицы: 23.19 МБ
Подготовка недель: 0.30 сек.


Сохраняем недельную таблицу в Parquet отдельно от первого эксперимента. Повторяющиеся идентификаторы храним как категории, чтобы уменьшить расход памяти.

In [5]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
weekly_path = DATA_PROCESSED / f"m5_{STORE_ID.lower()}_all_weekly.parquet"
weekly.to_parquet(weekly_path, index=False)

display(
    weekly.groupby("cat_id", observed=True)
    .agg(
        items=("item_id", "nunique"),
        weekly_rows=("weekly_sales", "size"),
        mean_weekly_sales=("weekly_sales", "mean"),
    )
    .round(2)
)
print(f"Сохранено: {weekly_path.name}")

,items,weekly_rows,mean_weekly_sales
cat_id,,,
FOODS,1437,392301,13.70
HOBBIES,565,154245,5.68
HOUSEHOLD,1047,285831,5.04


Сохранено: m5_ca_1_all_weekly.parquet


<a id="features"></a>

## 3. Признаки и периоды проверки

Начинаем с признаков первого ноутбука: продажи 1–4 недели назад, средние за 4 и 8 недель, стандартное отклонение за 4 недели, месяц и номер недели. Скользящие признаки считаем после `shift(1)`, поэтому продажи прогнозируемой недели в них не входят.

| Часть исходной истории | Назначение |
|---|---|
| До 31 января 2016 | Обучение первой модели |
| 1 февраля — 27 марта 2016 | Первые сравнения моделей и признаков: 8 недель |
| 28 марта — 24 апреля 2016 | `reference`: этот период уже смотрели в первом ноутбуке |

Позже добавляем четыре временных окна для сравнения вариантов. Финальная проверка проводится отдельно: **25 апреля — 22 мая 2016**, после выбора параметров.

In [6]:
started = perf_counter()
model_data = split_by_time(add_weekly_features(weekly))
feature_seconds = perf_counter() - started

split_summary = model_data.groupby("split", observed=True).agg(
    first_week_end=("week_end", "min"),
    last_week_end=("week_end", "max"),
    weeks=("week_end", "nunique"),
    rows=("weekly_sales", "size"),
)

display(split_summary)
print(f"Подготовка признаков: {feature_seconds:.2f} сек.")
print(f"Память таблицы с признаками: {model_data.memory_usage(deep=True).sum() / 1024**2:.2f} МБ")

,first_week_end,last_week_end,weeks,rows
split,,,,
train,2011-02-06,2016-01-31,261,795789
validation,2016-02-07,2016-03-27,8,24392
reference,2016-04-03,2016-04-24,4,12196


Подготовка признаков: 3.27 сек.
Память таблицы с признаками: 96.22 МБ


Для обучения и оценки оставляем недели, к началу которых у товара уже была запись о цене. Нулевые продажи после этой даты сохраняем. В начале истории также нужны восемь прошлых недель для `mean_8w`.

Если первая цена появилась только в течение прогнозируемой недели, эту строку исключаем. Таким образом, здесь проверяем прогноз товаров с доступной историей; запуск новых товаров остаётся отдельной задачей.

In [7]:
has_history = model_data[NUMERIC_FEATURES].notna().all(axis=1)
train_mask = model_data["split"].eq("train") & model_data["known_at_origin"] & has_history
validation_candidates = model_data["split"].eq("validation")
validation_mask = validation_candidates & model_data["known_at_origin"]
validation_data = model_data.loc[validation_mask].copy()

assert validation_data[FEATURE_COLUMNS].notna().all().all()
assert validation_data[list(BASELINE_METHODS.values())].notna().all().all()

print(f"Строк для будущего обучения: {train_mask.sum():,}")
print(f"Всего строк валидации: {validation_candidates.sum():,}")
print(f"Из них с известной ранее ценой: {len(validation_data):,}")
print(
    f"Исключено до первой известной цены: {(validation_candidates & ~model_data['known_at_origin']).sum():,}"
)

display(
    validation_data[
        ["week_end", "item_id", "cat_id", "lag_1w", "mean_4w", "mean_8w", "weekly_sales"]
    ]
    .head(10)
    .round(2)
)

Строк для будущего обучения: 623,292
Всего строк валидации: 24,392
Из них с известной ранее ценой: 24,390
Исключено до первой известной цены: 2


/var/folders/_5/xfqsygpn717bwxf2_1nbmcvh0000gn/T/ipykernel_54567/384630940.py:17: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].head(10).round(2))


,week_end,item_id,cat_id,lag_1w,mean_4w,mean_8w,weekly_sales
261,2016-02-07,FOODS_1_001,FOODS,7.0,3.75,4.62,8
262,2016-02-14,FOODS_1_001,FOODS,8.0,4.75,4.00,6
263,2016-02-21,FOODS_1_001,FOODS,6.0,5.50,4.38,5
264,2016-02-28,FOODS_1_001,FOODS,5.0,6.50,4.50,1
265,2016-03-06,FOODS_1_001,FOODS,1.0,5.00,4.38,4
266,2016-03-13,FOODS_1_001,FOODS,4.0,4.00,4.38,4
267,2016-03-20,FOODS_1_001,FOODS,4.0,3.50,4.50,10
268,2016-03-27,FOODS_1_001,FOODS,10.0,4.75,5.62,6
534,2016-02-07,FOODS_1_002,FOODS,1.0,1.50,2.75,8
535,2016-02-14,FOODS_1_002,FOODS,8.0,3.00,3.12,3


<a id="baselines"></a>

## 4. Простые прогнозы и метрики

Сравниваем продажи предыдущей недели, среднее и медиану за четыре прошлые недели. Все методы оцениваем на одинаковых строках.

| Метрика | Что измеряет |
|---|---|
| MAE | Среднюю абсолютную ошибку в единицах товара за неделю |
| WAPE | Сумму абсолютных ошибок, делённую на суммарные фактические продажи |
| Bias | Суммарную разницу «прогноз − факт», делённую на суммарные продажи |

У MAE и WAPE меньшее значение лучше. Для Bias смотрим близость к нулю: отрицательный знак означает недопрогноз общего объёма. WAPE не показывает процент точных прогнозов.

Состав данных отличается от первого ноутбука: здесь есть все три категории. Поэтому сравниваем методы внутри одной выборки, а не абсолютный MAE разных наборов товаров.

In [8]:
baseline_results = evaluate_forecasts(validation_data, BASELINE_METHODS)
display(
    baseline_results.style.format(
        {
            "MAE": "{:.2f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

,MAE,WAPE,Bias
Метод,,,
Предыдущая неделя,4.00,39.33%,-1.16%
Среднее за 4 недели,3.56,34.98%,-2.07%
Медиана за 4 недели,3.62,35.57%,-4.27%


Проверяем те же простые методы отдельно для FOODS, HOBBIES и HOUSEHOLD. У категорий разные объёмы продаж: MAE показывает ошибку в штуках, WAPE помогает оценить её относительно общего объёма категории.

In [9]:
category_results = pd.concat(
    {
        category: evaluate_forecasts(group, BASELINE_METHODS)
        for category, group in validation_data.groupby("cat_id", observed=True)
    },
    names=["category", "Метод"],
)

display(
    category_results.style.format(
        {
            "MAE": "{:.2f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

Сохраняем метрики простых методов и сведения о размере данных в `results/ca1_scaling`. Следующие модели будут сравниваться с этим исходным результатом.

In [10]:
scaling_results_dir = RESULTS / "ca1_scaling"
scaling_results_dir.mkdir(parents=True, exist_ok=True)
baseline_results.to_csv(scaling_results_dir / "baseline_metrics.csv", index_label="Метод")
category_results.to_csv(scaling_results_dir / "baseline_metrics_by_category.csv")
category_items.to_csv(scaling_results_dir / "category_items.csv")

scaling_summary = {
    "store_id": STORE_ID,
    "items": len(sales),
    "daily_history_days": len(day_columns),
    "complete_weeks": int(weekly["week_end"].nunique()),
    "weekly_rows": len(weekly),
    "training_rows_eligible": int(train_mask.sum()),
    "validation_rows_available": int(validation_candidates.sum()),
    "validation_rows_evaluated": len(validation_data),
    "selection_rule": "price registered by forecast origin; historical zeros retained",
    "reference_period_already_viewed_in_pilot": True,
    "weekly_memory_mb": float(weekly.memory_usage(deep=True).sum() / 1024**2),
    "feature_memory_mb": float(model_data.memory_usage(deep=True).sum() / 1024**2),
    "load_seconds": load_seconds,
    "weekly_seconds": weekly_seconds,
    "feature_seconds": feature_seconds,
    "catboost_trained": False,
}
(scaling_results_dir / "data_summary.json").write_text(
    json.dumps(scaling_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Результаты: {scaling_results_dir}")

Результаты: /Users/daniilsestov/Desktop/pet_projects/demand-planner/results/ca1_scaling


На первых восьми неделях проверки осталось 24 390 строк из 24 392. У двух строк FOODS_3_595 первая цена ещё не была зарегистрирована к началу прогноза. Правило исключения задано по доступности цены, а не по будущим продажам.

Лучший простой метод на этом периоде — среднее за четыре недели: **MAE 3.5586, WAPE 34.98%**. Он становится основным ориентиром для CatBoost. Эти значения относятся к февралю–марту; финальные апрель–май оцениваются отдельно.

<a id="catboost"></a>

## 5. Первый CatBoost

Обучаем одну общую модель для всех 3 049 товаров CA_1. Начальный набор содержит идентификаторы товара и магазина, лаги продаж, скользящие статистики и два календарных признака.

Сначала готовим `X_train`, `X_valid` и целевые продажи. Идентификаторы передаём CatBoost как категориальные признаки.

In [11]:
import numpy as np
from catboost import CatBoostRegressor
from src.features import CATEGORICAL_FEATURES

X_train = model_data.loc[train_mask, FEATURE_COLUMNS].copy()
y_train = model_data.loc[train_mask, "weekly_sales"]

X_valid = validation_data[FEATURE_COLUMNS].copy()
y_valid = validation_data["weekly_sales"]

assert X_train.notna().all().all()
assert X_valid.notna().all().all()
assert model_data.loc[train_mask, "week_end"].max() < validation_data["week_end"].min()

for frame in (X_train, X_valid):
    for column in CATEGORICAL_FEATURES:
        frame[column] = frame[column].astype(str).astype(object)

print(f"Строк обучения: {len(X_train):,}")
print(f"Строк валидации: {len(X_valid):,}")

Строк обучения: 623,292
Строк валидации: 24,390


### 5.1. Обучение и ранняя остановка

Лимит — 5 000 деревьев. Обучение останавливается после 100 итераций без улучшения MAE на валидации; сохраняется модель до лучшей итерации. `test` в логе CatBoost здесь означает валидационную выборку.

Первый запуск с лимитом 1 000 закончился на последнем дереве. После увеличения лимита модель смогла выбрать число деревьев по валидации.

In [12]:
model = CatBoostRegressor(
    iterations=5000,
    depth=6,
    learning_rate=0.05,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=RANDOM_STATE,
    thread_count=4,
    allow_writing_files=False,
)

model.fit(
    X_train,
    y_train,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100,
    use_best_model=True,
    verbose=250,
)

print(f"Деревьев в модели: {model.tree_count_}")

0:	learn: 9.7751480	test: 8.4068994	best: 8.4068994 (0)	total: 148ms	remaining: 12m 18s
250:	learn: 4.0587909	test: 3.4201749	best: 3.4201295 (247)	total: 16.1s	remaining: 5m 5s
500:	learn: 4.0432074	test: 3.4134053	best: 3.4134051 (499)	total: 30s	remaining: 4m 29s
750:	learn: 4.0293361	test: 3.3994857	best: 3.3994854 (748)	total: 43.1s	remaining: 4m 4s
1000:	learn: 3.9784435	test: 3.3780495	best: 3.3780495 (1000)	total: 56.4s	remaining: 3m 45s
1250:	learn: 3.9464084	test: 3.3735920	best: 3.3727405 (1187)	total: 1m 9s	remaining: 3m 29s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.372740497
bestIteration = 1187

Shrink model to first 1188 iterations.
Деревьев в модели: 1188


### 5.2. Общие ошибки

Получаем прогноз на тех же восьми неделях, что и у простых методов. Отрицательные прогнозы заменяем нулём, затем считаем MAE, WAPE и Bias.

In [13]:
validation_predictions = validation_data.copy()

validation_predictions["prediction_catboost"] = np.maximum(
    0,
    model.predict(X_valid),
)

methods_with_catboost = BASELINE_METHODS.copy()
methods_with_catboost["CatBoost"] = "prediction_catboost"

model_results = evaluate_forecasts(
    validation_predictions,
    methods_with_catboost,
)

display(
    model_results.style.format(
        {
            "MAE": "{:.3f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

baseline_mae = model_results.loc["Среднее за 4 недели", "MAE"]
catboost_mae = model_results.loc["CatBoost", "MAE"]
relative_gain = (baseline_mae - catboost_mae) / baseline_mae

print(f"Снижение MAE относительно среднего 4 недель: {relative_gain:+.2%}")

,MAE,WAPE,Bias
Метод,,,
Предыдущая неделя,4.001,39.33%,-1.16%
Среднее за 4 недели,3.559,34.98%,-2.07%
Медиана за 4 недели,3.618,35.57%,-4.27%
CatBoost,3.372,33.15%,-6.26%


Снижение MAE относительно среднего 4 недель: +5.24%


### 5.3. Ошибки по категориям

Сравниваем исходный CatBoost со средним четырёх недель отдельно в каждой категории. Так видно, на каких товарах появляется выигрыш.

In [14]:
catboost_category_results = pd.concat(
    {
        category: evaluate_forecasts(group, methods_with_catboost)
        for category, group in validation_predictions.groupby("cat_id", observed=True)
    },
    names=["category", "Метод"],
)

display(
    catboost_category_results.style.format(
        {
            "MAE": "{:.3f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

Исходный CatBoost сохранил **1 188 деревьев**. На первых восьми неделях получил MAE 3.372, WAPE 33.15% и Bias −6.26%. Снижение MAE относительно среднего за четыре недели — 5.24%.

Увеличение лимита деревьев помогло немного. Следующий шаг — проверить дополнительную информацию о товарах и прошлых продажах.

<a id="feature-experiments"></a>

## 6. Новые признаки

Даты и строки первого сравнения сохраняем. Категории и длинную историю добавляем последовательно. Затем к модели «категории + история» по отдельности добавляем ещё три блока, а после проверяем их совместно.

### 6.1. Категории товара

Передаём `cat_id` и `dept_id` как категориальные признаки: модель получает категорию и отдел товара. Остальные признаки и параметры сохраняем.

In [15]:
extra_categories = ["cat_id", "dept_id"]

features_categories = [*FEATURE_COLUMNS, *extra_categories]
categorical_categories = [*CATEGORICAL_FEATURES, *extra_categories]

X_train_categories = model_data.loc[train_mask, features_categories].copy()

X_valid_categories = validation_data[features_categories].copy()

for frame in (X_train_categories, X_valid_categories):
    for column in categorical_categories:
        frame[column] = frame[column].astype(str).astype(object)

assert X_train_categories.index.equals(y_train.index)
assert X_valid_categories.index.equals(y_valid.index)

category_params = model.get_params().copy()
category_params["iterations"] = 5000

model_categories = CatBoostRegressor(**category_params)
print(f"Лимит итераций: {model_categories.get_params()['iterations']}")

model_categories.fit(
    X_train_categories,
    y_train,
    cat_features=categorical_categories,
    eval_set=(X_valid_categories, y_valid),
    early_stopping_rounds=100,
    use_best_model=True,
    verbose=250,
)

print(f"Деревьев в модели: {model_categories.tree_count_}")

Лимит итераций: 5000
0:	learn: 9.8212731	test: 8.4485598	best: 8.4485598 (0)	total: 102ms	remaining: 8m 31s
250:	learn: 4.1059017	test: 3.4258505	best: 3.4258029 (240)	total: 18.9s	remaining: 5m 57s
500:	learn: 4.0494692	test: 3.3940208	best: 3.3940208 (500)	total: 34.5s	remaining: 5m 9s
750:	learn: 3.9856226	test: 3.3755916	best: 3.3752770 (749)	total: 51.8s	remaining: 4m 52s
1000:	learn: 3.9538363	test: 3.3654827	best: 3.3654323 (982)	total: 1m 9s	remaining: 4m 38s
1250:	learn: 3.9085883	test: 3.3541144	best: 3.3540950 (1230)	total: 1m 28s	remaining: 4m 26s
1500:	learn: 3.8797305	test: 3.3497580	best: 3.3490570 (1470)	total: 1m 48s	remaining: 4m 13s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.34812512
bestIteration = 1612

Shrink model to first 1613 iterations.
Деревьев в модели: 1613


Считаем прогноз модели с категориями и сравниваем его с исходным CatBoost и простыми методами. Состав проверочных строк остаётся прежним.

In [16]:
category_predictions = validation_predictions.copy()

category_predictions["prediction_categories"] = np.maximum(
    0,
    model_categories.predict(X_valid_categories),
)

category_methods = {
    "Среднее за 4 недели": "prediction_mean_4w",
    "CatBoost — исходный": "prediction_catboost",
    "CatBoost + категории": "prediction_categories",
}

category_comparison = evaluate_forecasts(
    category_predictions,
    category_methods,
)

display(
    category_comparison.style.format(
        {
            "MAE": "{:.4f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

old_mae = category_comparison.loc["CatBoost — исходный", "MAE"]
new_mae = category_comparison.loc["CatBoost + категории", "MAE"]
gain = (old_mae - new_mae) / old_mae

print(f"Снижение MAE после добавления категорий: {gain:+.2%}")

,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
CatBoost — исходный,3.3722,33.15%,-6.26%
CatBoost + категории,3.3471,32.90%,-6.12%


Снижение MAE после добавления категорий: +0.74%


### 6.2. История и направление изменений

Добавляем четыре признака:

- `mean_2w` — средние продажи последних двух недель;
- `mean_13w` — средние продажи последних тринадцати недель;
- `trend_2w_8w` — разница средних за две и восемь недель;
- `std_8w` — стандартное отклонение продаж за восемь недель.

Перед расчётами стоит `shift(1)`: продажи прогнозируемой недели в признаки не попадают. Когда тринадцати прошлых недель ещё нет, оставляем `NaN`. Строки обучения сохраняем; CatBoost обрабатывает числовые пропуски.

Сравниваем новую модель с вариантом, в котором уже есть категории.

In [17]:
history_data = model_data.sort_values(["store_id", "item_id", "week_end"]).copy()

sales_groups = history_data.groupby(
    ["store_id", "item_id"],
    observed=True,
    sort=False,
)["weekly_sales"]

for window in (2, 13):
    history_data[f"mean_{window}w"] = sales_groups.transform(
        lambda series: series.shift(1).rolling(window).mean()
    )

history_data["std_8w"] = sales_groups.transform(lambda series: series.shift(1).rolling(8).std())

history_data["trend_2w_8w"] = history_data["mean_2w"] - history_data["mean_8w"]

new_history_features = [
    "mean_2w",
    "mean_13w",
    "trend_2w_8w",
    "std_8w",
]

features_history = [*features_categories, *new_history_features]

X_train_history = history_data.loc[X_train_categories.index, features_history].copy()

X_valid_history = history_data.loc[X_valid_categories.index, features_history].copy()

for frame in (X_train_history, X_valid_history):
    for column in categorical_categories:
        frame[column] = frame[column].astype(str).astype(object)

assert X_train_history.index.equals(y_train.index)
assert X_valid_history.index.equals(y_valid.index)

Обучаем модель с категориями и четырьмя признаками длинной истории. В начале истории числовые пропуски оставляем: они не меняют состав выбранных строк.

In [18]:
history_params = model_categories.get_params().copy()
history_params["iterations"] = 5000

model_history = CatBoostRegressor(**history_params)
print(f"Лимит итераций: {model_history.get_params()['iterations']}")

model_history.fit(
    X_train_history,
    y_train,
    cat_features=categorical_categories,
    eval_set=(X_valid_history, y_valid),
    early_stopping_rounds=100,
    use_best_model=True,
    verbose=250,
)

print(f"Деревьев в модели: {model_history.tree_count_}")

Лимит итераций: 5000
0:	learn: 9.8162009	test: 8.4439309	best: 8.4439309 (0)	total: 105ms	remaining: 8m 45s
250:	learn: 4.1243521	test: 3.4153579	best: 3.4153514 (239)	total: 19.4s	remaining: 6m 6s
500:	learn: 3.9838341	test: 3.3638258	best: 3.3638258 (500)	total: 38s	remaining: 5m 41s
750:	learn: 3.9486388	test: 3.3530238	best: 3.3523817 (723)	total: 56.9s	remaining: 5m 21s
1000:	learn: 3.9120279	test: 3.3412943	best: 3.3412943 (1000)	total: 1m 16s	remaining: 5m 6s
1250:	learn: 3.8855412	test: 3.3383434	best: 3.3378760 (1187)	total: 1m 36s	remaining: 4m 50s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.33787596
bestIteration = 1187

Shrink model to first 1188 iterations.
Деревьев в модели: 1188


Добавляем прогноз модели с историей в общую таблицу и считаем её прирост относительно исходного CatBoost.

In [19]:
history_predictions = category_predictions.copy()

assert history_predictions.index.equals(X_valid_history.index)

history_predictions["prediction_history"] = np.maximum(
    0,
    model_history.predict(X_valid_history),
)

history_methods = {
    "Среднее за 4 недели": "prediction_mean_4w",
    "CatBoost + категории": "prediction_categories",
    "CatBoost + категории + история": "prediction_history",
}

history_comparison = evaluate_forecasts(
    history_predictions,
    history_methods,
)

display(
    history_comparison.style.format(
        {
            "MAE": "{:.4f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

old_mae = history_comparison.loc["CatBoost + категории", "MAE"]
new_mae = history_comparison.loc["CatBoost + категории + история", "MAE"]

gain = (old_mae - new_mae) / old_mae
print(f"Снижение MAE после добавления истории: {gain:+.2%}")

,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
CatBoost + категории,3.3471,32.90%,-6.12%
CatBoost + категории + история,3.3374,32.81%,-6.49%


Снижение MAE после добавления истории: +0.29%


### 6.3. Общая функция для отдельных экспериментов

Модель «категории + история» служит основой. К ней отдельно добавляем признаки нулевых продаж, календаря и цены. Строки таблицы с `+` до совместного запуска относятся к разным моделям.

`run_feature_block` выбирает нужные столбцы, обучает CatBoost на прежних строках и добавляет прогноз в общую таблицу. Лимит — 5 000 деревьев, ранняя остановка — 100 итераций. Прирост считаем относительно модели с категориями и историей.

In [20]:
experiment_data = history_data.copy()
experiment_predictions = history_predictions.copy()

experiment_methods = {
    "Среднее за 4 недели": "prediction_mean_4w",
    "Категории + история": "prediction_history",
}


def run_feature_block(name, added_features):
    columns = [*features_history, *added_features]
    categorical = [
        column for column in [*categorical_categories, "week_events"] if column in columns
    ]

    X_train_block = experiment_data.loc[X_train_history.index, columns].copy()

    X_valid_block = experiment_data.loc[X_valid_history.index, columns].copy()

    for frame in (X_train_block, X_valid_block):
        for column in categorical:
            frame[column] = frame[column].astype(str).astype(object)

    assert X_train_block.index.equals(y_train.index)
    assert X_valid_block.index.equals(y_valid.index)
    assert experiment_predictions.index.equals(X_valid_block.index)

    params = model_history.get_params().copy()
    params["iterations"] = 5000

    candidate = CatBoostRegressor(**params)
    candidate.fit(
        X_train_block,
        y_train,
        cat_features=categorical,
        eval_set=(X_valid_block, y_valid),
        early_stopping_rounds=100,
        use_best_model=True,
        verbose=500,
    )

    experiment_predictions[name] = np.maximum(0, candidate.predict(X_valid_block))
    experiment_methods[name] = name

    results = evaluate_forecasts(experiment_predictions, experiment_methods)

    display(
        results.style.format(
            {
                "MAE": "{:.4f}",
                "WAPE": "{:.2%}",
                "Bias": "{:+.2%}",
            }
        )
    )

    base_mae = results.loc["Категории + история", "MAE"]
    gain = 1 - results.loc[name, "MAE"] / base_mae

    print(f"Снижение MAE относительно модели с историей: {gain:+.2%}")
    print(f"Деревьев сохранено: {candidate.tree_count_}")

    return candidate

### 6.4. Нулевые продажи

Описываем, насколько регулярно покупают товар:

- `zero_share_8w` — доля нулевых недель среди последних восьми;
- `zero_run` — число последних недель подряд без продаж;
- `mean_nonzero_8w` — средние продажи в ненулевые недели за то же окно.

Все значения считаются по прошлым продажам. Если за восемь недель положительных продаж не было, среднее по ненулевым неделям остаётся `NaN`. Причину отсутствия продаж эти признаки не определяют.

In [21]:
def zero_share(series):
    past = series.shift(1)
    zeros = past.eq(0).astype(float).where(past.notna())
    return zeros.rolling(8).mean()


def zero_run(series):
    zeros = series.shift(1).eq(0)
    runs = (~zeros).cumsum()
    return zeros.groupby(runs).cumsum()


sales_groups = experiment_data.groupby(
    ["store_id", "item_id"],
    observed=True,
    sort=False,
)["weekly_sales"]

experiment_data["zero_share_8w"] = sales_groups.transform(zero_share)
experiment_data["zero_run"] = sales_groups.transform(zero_run)

experiment_data["mean_nonzero_8w"] = sales_groups.transform(
    lambda series: (series.shift(1).replace(0, np.nan).rolling(8, min_periods=1).mean())
)

zero_features = [
    "zero_share_8w",
    "zero_run",
    "mean_nonzero_8w",
]

model_zeros = run_feature_block("+ нулевые продажи", zero_features)

0:	learn: 9.8379520	test: 8.4629858	best: 8.4629858 (0)	total: 103ms	remaining: 8m 33s
500:	learn: 4.0428246	test: 3.3777779	best: 3.3777767 (499)	total: 44.1s	remaining: 6m 36s
1000:	learn: 3.9542583	test: 3.3534882	best: 3.3534882 (1000)	total: 1m 21s	remaining: 5m 25s
1500:	learn: 3.8880475	test: 3.3366246	best: 3.3366246 (1498)	total: 2m 3s	remaining: 4m 48s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.333938337
bestIteration = 1613

Shrink model to first 1614 iterations.


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
Категории + история,3.3374,32.81%,-6.49%
+ нулевые продажи,3.3331,32.77%,-6.44%


Снижение MAE относительно модели с историей: +0.13%
Деревьев сохранено: 1614


### 6.5. События календаря и SNAP

Для прогнозируемой недели собираем:

- `snap_days` — число дней с флагом `snap_CA`;
- `event_days` — число дней хотя бы с одним событием;
- `week_events` — названия событий за неделю, категориальный признак.

Эта информация берётся из календаря и известна заранее. Дневные признаки собираем в недельные, поскольку прогнозируем сумму продаж с понедельника по воскресенье.

In [22]:
calendar_days = calendar.copy()

calendar_days["week_end"] = calendar_days["date"] + pd.to_timedelta(
    6 - calendar_days["date"].dt.dayofweek,
    unit="D",
)

calendar_days["has_event"] = (
    calendar_days["event_name_1"].notna() | calendar_days["event_name_2"].notna()
).astype("int8")

calendar_weekly = calendar_days.groupby("week_end").agg(
    snap_days=("snap_CA", "sum"),
    event_days=("has_event", "sum"),
)

events = calendar_days.melt(
    id_vars="week_end",
    value_vars=["event_name_1", "event_name_2"],
    value_name="event",
).dropna(subset=["event"])

calendar_weekly["week_events"] = events.groupby("week_end")["event"].agg(
    lambda names: " | ".join(sorted(names.unique()))
)
calendar_weekly["week_events"] = calendar_weekly["week_events"].fillna("NoEvent")

calendar_features = ["snap_days", "event_days", "week_events"]

experiment_data = experiment_data.join(
    calendar_weekly[calendar_features],
    on="week_end",
    validate="many_to_one",
)

model_calendar = run_feature_block("+ календарь и SNAP", calendar_features)

0:	learn: 9.8267904	test: 8.4535844	best: 8.4535844 (0)	total: 120ms	remaining: 9m 59s
500:	learn: 3.9790332	test: 3.3795263	best: 3.3794671 (457)	total: 50.9s	remaining: 7m 37s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.379467058
bestIteration = 457

Shrink model to first 458 iterations.


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
Категории + история,3.3374,32.81%,-6.49%
+ нулевые продажи,3.3331,32.77%,-6.44%
+ календарь и SNAP,3.3787,33.21%,-6.84%


Снижение MAE относительно модели с историей: -1.24%
Деревьев сохранено: 458


### 6.6. История цены

Ценовая неделя M5 идёт с субботы по пятницу. Для прогноза, который делаем в воскресенье, используем цены уже завершившихся периодов. Сдвиг на девять дней связывает пятницу ценового периода с воскресеньем прогнозируемой недели.

Добавляем:

- `price_last` — последнюю известную цену завершённого периода;
- `price_change_1w` — относительное изменение этой цены за неделю;
- `price_relative_4w` — отклонение от средней цены за четыре периода;
- `price_missing_last_period` — отсутствие записи о цене за последний завершённый период.

Пропуски цены заполняем только предыдущими известными значениями через `ffill`. Отсутствие записи о цене не означает отсутствие товара на складе.

In [23]:
series_keys = ["store_id", "item_id"]

price_week_ends = calendar.groupby("wm_yr_wk")["date"].min() + pd.Timedelta(days=6)

price_lookup = prices[[*series_keys, "wm_yr_wk", "sell_price"]].copy()

price_lookup["week_end"] = price_lookup["wm_yr_wk"].map(price_week_ends) + pd.Timedelta(days=9)

price_lookup = price_lookup.rename(columns={"sell_price": "price_quote"}).set_index(
    [*series_keys, "week_end"]
)

experiment_data = experiment_data.join(
    price_lookup[["price_quote"]],
    on=[*series_keys, "week_end"],
    validate="many_to_one",
)

experiment_data["price_missing_last_period"] = experiment_data["price_quote"].isna().astype("int8")

experiment_data["price_last"] = experiment_data.groupby(
    series_keys,
    observed=True,
    sort=False,
)["price_quote"].ffill()

price_groups = experiment_data.groupby(
    series_keys,
    observed=True,
    sort=False,
)["price_last"]

previous_price = price_groups.shift(1)
mean_price_4w = price_groups.transform(lambda series: series.rolling(4, min_periods=1).mean())

experiment_data["price_change_1w"] = experiment_data["price_last"] / previous_price - 1
experiment_data["price_relative_4w"] = experiment_data["price_last"] / mean_price_4w - 1

price_features = [
    "price_last",
    "price_change_1w",
    "price_relative_4w",
    "price_missing_last_period",
]

model_price = run_feature_block("+ история цены", price_features)

0:	learn: 9.8125777	test: 8.4399539	best: 8.4399539 (0)	total: 104ms	remaining: 8m 39s
500:	learn: 3.9716664	test: 3.3668528	best: 3.3668528 (500)	total: 40s	remaining: 5m 58s
1000:	learn: 3.8969480	test: 3.3413291	best: 3.3413291 (1000)	total: 1m 21s	remaining: 5m 24s
1500:	learn: 3.8540552	test: 3.3335833	best: 3.3327510 (1420)	total: 2m 2s	remaining: 4m 46s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.332751016
bestIteration = 1420

Shrink model to first 1421 iterations.


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
Категории + история,3.3374,32.81%,-6.49%
+ нулевые продажи,3.3331,32.77%,-6.44%
+ календарь и SNAP,3.3787,33.21%,-6.84%
+ история цены,3.3322,32.76%,-6.44%


Снижение MAE относительно модели с историей: +0.15%
Деревьев сохранено: 1421


### 6.7. Результат отдельных экспериментов

| Вариант | MAE | Снижение MAE относительно модели с историей |
|---|---:|---:|
| Категории + история | 3.3374 | — |
| + нулевые продажи | 3.3331 | +0.13% |
| + календарь и SNAP | 3.3787 | −1.24% |
| + история цены | 3.3322 | +0.15% |

Нулевые продажи и цена дали небольшое улучшение. Календарь с SNAP отдельно ухудшил MAE. Результаты относятся к одному временному периоду; полезность общего набора проверяем следующим запуском.

### 6.8. Все признаки вместе

Объединяем категории, историю, нулевые продажи, календарь и цену. Получаем **27 признаков**, из них пять категориальных. Это отдельный совместный эксперимент: эффект нескольких блоков не обязан совпадать с суммой отдельных улучшений.

In [24]:
all_added_features = list(
    dict.fromkeys(
        [
            *zero_features,
            *calendar_features,
            *price_features,
        ]
    )
)

features_all = [*features_history, *all_added_features]

print(f"Всего признаков: {len(features_all)}")

model_all_features = run_feature_block(
    "+ все признаки",
    all_added_features,
)

Всего признаков: 27
0:	learn: 9.8217103	test: 8.4476763	best: 8.4476763 (0)	total: 123ms	remaining: 10m 14s
500:	learn: 3.9490344	test: 3.3453014	best: 3.3453014 (500)	total: 52.5s	remaining: 7m 51s
1000:	learn: 3.8511589	test: 3.3209769	best: 3.3209769 (1000)	total: 1m 46s	remaining: 7m 6s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.320323451
bestIteration = 1126

Shrink model to first 1127 iterations.


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.5586,34.98%,-2.07%
Категории + история,3.3374,32.81%,-6.49%
+ нулевые продажи,3.3331,32.77%,-6.44%
+ календарь и SNAP,3.3787,33.21%,-6.84%
+ история цены,3.3322,32.76%,-6.44%
+ все признаки,3.3192,32.63%,-5.39%


Снижение MAE относительно модели с историей: +0.54%
Деревьев сохранено: 1127


Совместная модель получила MAE **3.3192**, WAPE **32.63%** и Bias **−5.39%** на первых восьми неделях. Дальше оцениваем её на нескольких периодах, чтобы не выбирать модель только по одному удачному отрезку.

<a id="rolling-validation"></a>

## 7. Четыре временных окна

Проверяем общую модель на четырёх восьминедельных периодах: май–июнь, август–сентябрь, ноябрь–декабрь 2015 года и февраль–март 2016 года.

Для каждого окна используем два этапа:

1. На восьми предшествующих неделях выбираем число деревьев через раннюю остановку.
2. С выбранным числом деревьев обучаем новую модель на всей истории до начала проверочного окна.

Продажи проверочного окна не используются для выбора числа деревьев. Внутри окна модель остаётся фиксированной, а лаги обновляются после завершения каждой недели.

In [25]:
CV_WEEKS = 8
STOP_WEEKS = 8
CV_OFFSETS = (39, 26, 13, 0)

cv_data = experiment_data
cv_X = cv_data[features_all].copy()
cv_y = cv_data["weekly_sales"]

cv_categorical = [
    column for column in [*categorical_categories, "week_events"] if column in features_all
]

for column in cv_categorical:
    cv_X[column] = cv_X[column].astype(str).astype(object)

cv_eligible = cv_data["known_at_origin"] & cv_data[NUMERIC_FEATURES].notna().all(axis=1)

last_validation_week = cv_data.loc[cv_data["split"].eq("validation"), "week_end"].max()

cv_weeks = (
    cv_data.loc[cv_data["week_end"].le(last_validation_week), "week_end"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

cv_windows = []

for fold, offset in enumerate(CV_OFFSETS, start=1):
    end = len(cv_weeks) - offset
    assert end > CV_WEEKS + STOP_WEEKS

    cv_windows.append(
        {
            "fold": fold,
            "tune_start": cv_weeks[end - CV_WEEKS - STOP_WEEKS],
            "score_start": cv_weeks[end - CV_WEEKS],
            "score_end": cv_weeks[end - 1],
        }
    )

display(pd.DataFrame(cv_windows).set_index("fold"))

,tune_start,score_start,score_end
fold,,,
1,2015-03-15,2015-05-10,2015-06-28
2,2015-06-14,2015-08-09,2015-09-27
3,2015-09-13,2015-11-08,2015-12-27
4,2015-12-13,2016-02-07,2016-03-27


### 7.1. Обучение в каждом окне

Цикл выполняет четыре пары обучений. Внешний период нужен только для оценки; `eval_set` содержит предшествующие недели. Прогнозы каждого окна сохраняем для последующего сравнения без повторного обучения контроля.

In [26]:
cv_params = model_all_features.get_params().copy()
cv_params["iterations"] = 5000
cv_params["allow_writing_files"] = False

cv_methods = {
    **BASELINE_METHODS,
    "Среднее за 8 недель": "mean_8w",
    "Среднее за 13 недель": "mean_13w",
    "CatBoost — все признаки": "prediction_cv",
}

score_columns = [
    "store_id",
    "item_id",
    "cat_id",
    "week_end",
    "weekly_sales",
    *BASELINE_METHODS.values(),
    "mean_8w",
    "mean_13w",
]

cv_parts = []
cv_fold_rows = []

for window in cv_windows:
    fold = window["fold"]
    dates = cv_data["week_end"]

    inner_train_mask = cv_eligible & dates.lt(window["tune_start"])

    inner_stop_mask = cv_eligible & dates.ge(window["tune_start"]) & dates.lt(window["score_start"])

    past_mask = cv_eligible & dates.lt(window["score_start"])

    score_mask = cv_eligible & dates.ge(window["score_start"]) & dates.le(window["score_end"])

    assert all(mask.any() for mask in (inner_train_mask, inner_stop_mask, past_mask, score_mask))

    print(
        f"\nОкно {fold}: " f"{window['score_start']:%Y-%m-%d} — " f"{window['score_end']:%Y-%m-%d}"
    )

    selector = CatBoostRegressor(**cv_params)

    selector.fit(
        cv_X.loc[inner_train_mask],
        cv_y.loc[inner_train_mask],
        cat_features=cv_categorical,
        eval_set=(
            cv_X.loc[inner_stop_mask],
            cv_y.loc[inner_stop_mask],
        ),
        early_stopping_rounds=100,
        use_best_model=True,
        verbose=500,
    )

    selected_trees = int(selector.tree_count_)
    print(f"Выбрано деревьев: {selected_trees}. Переобучаем на всём прошлом.")

    refit_params = cv_params.copy()
    refit_params["iterations"] = selected_trees

    cv_model = CatBoostRegressor(**refit_params)

    cv_model.fit(
        cv_X.loc[past_mask],
        cv_y.loc[past_mask],
        cat_features=cv_categorical,
        use_best_model=False,
        verbose=False,
    )

    fold_predictions = cv_data.loc[score_mask, score_columns].copy()
    fold_predictions["prediction_cv"] = np.maximum(0, cv_model.predict(cv_X.loc[score_mask]))
    fold_predictions["fold"] = fold

    fold_metrics = evaluate_forecasts(fold_predictions, cv_methods)

    baseline_mae = fold_metrics.loc["Среднее за 4 недели", "MAE"]
    model_mae = fold_metrics.loc["CatBoost — все признаки", "MAE"]

    cv_fold_rows.append(
        {
            **window,
            "trees": selected_trees,
            "rows": len(fold_predictions),
            "MAE_mean_4w": baseline_mae,
            "MAE_CatBoost": model_mae,
            "gain": 1 - model_mae / baseline_mae,
        }
    )

    cv_parts.append(fold_predictions)

    print(
        f"MAE: {model_mae:.4f}; "
        f"снижение относительно среднего 4 недель: "
        f"{1 - model_mae / baseline_mae:+.2%}"
    )


Окно 1: 2015-05-10 — 2015-06-28
0:	learn: 10.2204887	test: 8.4992853	best: 8.4992853 (0)	total: 118ms	remaining: 9m 48s
500:	learn: 4.0317963	test: 3.6498775	best: 3.6498775 (500)	total: 39.5s	remaining: 5m 54s
[Промежуточный журнал обучения сокращён; метрики и графики сохранены.]
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.613949534
bestIteration = 1121

Shrink model to first 1122 iterations.
Выбрано деревьев: 1122. Переобучаем на всём прошлом.
MAE: 3.5206; снижение относительно среднего 4 недель: +10.91%

Окно 2: 2015-08-09 — 2015-09-27
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.579604631
bestIteration = 1602

Shrink model to first 1603 iterations.
Выбрано деревьев: 1603. Переобучаем на всём прошлом.
MAE: 3.4775; снижение относительно среднего 4 недель: +9.01%

Окно 3: 2015-11-08 — 2015-12-27
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.498277612
bestIteration = 1503

Shrink model to first 1504 iterations.
Вы

### 7.2. Сводный результат

Сравниваем модели по окнам и по всем проверочным строкам вместе. Общий WAPE и Bias считаем по объединённым прогнозам, а не как среднее процентов отдельных окон.

CatBoost до подбора параметров получил **MAE 3.4097, WAPE 33.67%, Bias −4.91%** и обошёл среднее за четыре недели во всех четырёх окнах.

In [27]:
cv_predictions = pd.concat(cv_parts, ignore_index=True)
cv_fold_summary = pd.DataFrame(cv_fold_rows).set_index("fold")

display(
    cv_fold_summary[
        [
            "score_start",
            "score_end",
            "trees",
            "MAE_mean_4w",
            "MAE_CatBoost",
            "gain",
        ]
    ].style.format(
        {
            "score_start": "{:%Y-%m-%d}",
            "score_end": "{:%Y-%m-%d}",
            "MAE_mean_4w": "{:.4f}",
            "MAE_CatBoost": "{:.4f}",
            "gain": "{:+.2%}",
        }
    )
)

cv_overall = evaluate_forecasts(cv_predictions, cv_methods)

display(
    cv_overall.sort_values("MAE").style.format(
        {
            "MAE": "{:.4f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

wins = cv_fold_summary["gain"].gt(0).sum()
print(f"CatBoost лучше среднего 4 недель в {wins} из {len(cv_windows)} окон")

,score_start,score_end,trees,MAE_mean_4w,MAE_CatBoost,gain
fold,,,,,,
1,2015-05-10,2015-06-28,1122,3.9517,3.5206,+10.91%
2,2015-08-09,2015-09-27,1603,3.8220,3.4775,+9.01%
3,2015-11-08,2015-12-27,1504,3.6250,3.3185,+8.45%
4,2016-02-07,2016-03-27,1918,3.5586,3.3228,+6.63%


,MAE,WAPE,Bias
Метод,,,
CatBoost — все признаки,3.4097,33.67%,-4.91%
Среднее за 4 недели,3.7391,36.92%,+0.33%
Медиана за 4 недели,3.7827,37.35%,-1.95%
Среднее за 8 недель,3.9946,39.45%,+0.69%
Предыдущая неделя,4.0562,40.05%,+0.47%
Среднее за 13 недель,4.2770,42.24%,+0.23%


CatBoost лучше среднего 4 недель в 4 из 4 окон


<a id="tuning"></a>

## 8. Подбор параметров

### 8.1. Четыре предварительных варианта

Сравниваем контроль, глубины 4 и 8 и усиленную регуляризацию `l2_leaf_reg=10`. Подбор проводим на внутреннем восьминедельном периоде последнего окна: **7 декабря 2015 — 31 января 2016**. Соответствующие даты окончания недель — 13 декабря — 31 января.

Для выбора параметров используем MAE. WAPE, Bias, число деревьев и время сохраняем дополнительно. Внешние февраль–март не передаём в `eval_set` этого подбора.

In [28]:
tuning_window = cv_windows[-1]
dates = cv_data["week_end"]

tuning_train_mask = cv_eligible & dates.lt(tuning_window["tune_start"])

tuning_valid_mask = (
    cv_eligible & dates.ge(tuning_window["tune_start"]) & dates.lt(tuning_window["score_start"])
)

base_params = model_all_features.get_params().copy()
base_params["iterations"] = 5000
base_params["allow_writing_files"] = False

actual_params = model_all_features.get_all_params()
base_depth = int(actual_params["depth"])
base_l2 = float(actual_params["l2_leaf_reg"])

tuning_configs = {
    "Контроль": {
        "depth": base_depth,
        "l2_leaf_reg": base_l2,
    },
    "Глубина 4": {
        "depth": 4,
        "l2_leaf_reg": base_l2,
    },
    "Глубина 8": {
        "depth": 8,
        "l2_leaf_reg": base_l2,
    },
    "Регуляризация 10": {
        "depth": base_depth,
        "l2_leaf_reg": 10,
    },
}

tuning_rows = []
tuning_models = {}

for name, changes in tuning_configs.items():
    params = {**base_params, **changes}
    started = perf_counter()

    print(f"\n{name}: {changes}, лимит деревьев — 5000")

    candidate = CatBoostRegressor(**params)
    candidate.fit(
        cv_X.loc[tuning_train_mask],
        cv_y.loc[tuning_train_mask],
        cat_features=cv_categorical,
        eval_set=(
            cv_X.loc[tuning_valid_mask],
            cv_y.loc[tuning_valid_mask],
        ),
        early_stopping_rounds=100,
        use_best_model=True,
        verbose=250,
    )

    check = cv_data.loc[tuning_valid_mask, ["weekly_sales"]].copy()

    check["prediction"] = np.maximum(
        0,
        candidate.predict(cv_X.loc[tuning_valid_mask]),
    )

    metrics = evaluate_forecasts(check, {"candidate": "prediction"}).loc["candidate"]

    tuning_rows.append(
        {
            "variant": name,
            **changes,
            "trees": candidate.tree_count_,
            **metrics.to_dict(),
            "seconds": perf_counter() - started,
        }
    )

    tuning_models[name] = candidate

tuning_table = pd.DataFrame(tuning_rows).set_index("variant").sort_values("MAE")

control_mae = tuning_table.loc["Контроль", "MAE"]
tuning_table["gain"] = 1 - tuning_table["MAE"] / control_mae

display(
    tuning_table.style.format(
        {
            "MAE": "{:.4f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
            "gain": "{:+.2%}",
            "seconds": "{:.0f}",
        }
    )
)

best_tuning_name = tuning_table.index[0]
best_tuning_params = tuning_models[best_tuning_name].get_params().copy()

print(f"Лучший вариант предварительного отбора: {best_tuning_name}")


Контроль: {'depth': 6, 'l2_leaf_reg': 3.0}, лимит деревьев — 5000
0:	learn: 9.9085427	test: 7.9843970	best: 7.9843970 (0)	total: 181ms	remaining: 15m 6s
250:	learn: 4.0369666	test: 3.3578554	best: 3.3572478 (241)	total: 25.6s	remaining: 8m 4s
[Промежуточный журнал обучения сокращён; метрики и графики сохранены.]
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.292726742
bestIteration = 1917

Shrink model to first 1918 iterations.

Глубина 4: {'depth': 4, 'l2_leaf_reg': 3.0}, лимит деревьев — 5000
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.383625549
bestIteration = 492

Shrink model to first 493 iterations.

Глубина 8: {'depth': 8, 'l2_leaf_reg': 3.0}, лимит деревьев — 5000
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.347033497
bestIteration = 160

Shrink model to first 161 iterations.

Регуляризация 10: {'depth': 6, 'l2_leaf_reg': 10}, лимит деревьев — 5000
1750:	learn: 3.8038130	test: 3.2902318	best: 3.2901532 (174

,depth,l2_leaf_reg,trees,MAE,WAPE,Bias,seconds,gain
variant,,,,,,,,
Регуляризация 10,6,10.000000,2290,3.2829,34.90%,-6.02%,259,+0.25%
Контроль,6,3.000000,1918,3.2910,34.98%,-6.00%,199,+0.00%
Глубина 8,8,3.000000,161,3.3464,35.57%,-5.93%,35,-1.68%
Глубина 4,4,3.000000,493,3.3810,35.94%,-5.87%,46,-2.73%


Лучший вариант предварительного отбора: Регуляризация 10


### 8.2. История Optuna

Переносим четыре уже посчитанных варианта в Optuna. Их результаты берём с полной точностью из таблицы. История хранится в `models/ca1_optuna.sqlite3`, повторный запуск не добавляет те же ручные варианты заново.

Ищем глубину 4–8, скорость обучения 0.03–0.10 и регуляризацию 1–20.

In [30]:
import optuna

from optuna.distributions import FloatDistribution, IntDistribution
from optuna.trial import TrialState, create_trial

study_path = PROJECT_ROOT / "models" / "ca1_optuna.sqlite3"
study_path.parent.mkdir(parents=True, exist_ok=True)

study = optuna.create_study(
    study_name="ca1_all_features_inner_8w",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=4,
    ),
    storage=f"sqlite:///{study_path}",
    load_if_exists=True,
)

search_space = {
    "depth": IntDistribution(4, 8),
    "learning_rate": FloatDistribution(0.03, 0.10, log=True),
    "l2_leaf_reg": FloatDistribution(1.0, 20.0, log=True),
}

imported_variants = {trial.user_attrs.get("manual_variant") for trial in study.trials}

for name, row in tuning_table.iterrows():
    if name in imported_variants:
        continue

    previous_params = tuning_models[name].get_params()

    study.add_trial(
        create_trial(
            params={
                "depth": int(row["depth"]),
                "learning_rate": float(previous_params["learning_rate"]),
                "l2_leaf_reg": float(row["l2_leaf_reg"]),
            },
            distributions=search_space,
            value=float(row["MAE"]),
            user_attrs={
                "manual_variant": str(name),
                "trees": int(row["trees"]),
                "WAPE": float(row["WAPE"]),
                "Bias": float(row["Bias"]),
                "seconds": float(row["seconds"]),
            },
        )
    )

print(f"Попыток в истории: {len(study.trials)}")

/Users/daniilsestov/Desktop/pet_projects/demand-planner/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-10-20 12:43:28,020] A new study created in RDB with name: ca1_all_features_inner_8w


Попыток в истории: 4


### 8.3. Шесть новых попыток

Общий бюджет — десять попыток, включая четыре ручные. Optuna добавляет шесть новых обучений на том же внутреннем периоде. Лимит каждого — 5 000 деревьев, ранняя остановка — 100 итераций.

Результаты завершённых попыток сохраняются в SQLite. После перезапуска ядра нужно восстановить данные и признаки; базу поиска можно использовать повторно. Бюджет в ячейке считается с учётом уже записанных попыток.

In [31]:
X_tuning_train = cv_X.loc[tuning_train_mask]
y_tuning_train = cv_y.loc[tuning_train_mask]

X_tuning_valid = cv_X.loc[tuning_valid_mask]
y_tuning_valid = cv_y.loc[tuning_valid_mask]


def objective(trial):
    params = {
        **base_params,
        "iterations": 5000,
        "allow_writing_files": False,
        "depth": trial.suggest_int("depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.10, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
    }

    started = perf_counter()
    candidate = CatBoostRegressor(**params)

    candidate.fit(
        X_tuning_train,
        y_tuning_train,
        cat_features=cv_categorical,
        eval_set=(X_tuning_valid, y_tuning_valid),
        early_stopping_rounds=100,
        use_best_model=True,
        verbose=500,
    )

    check = cv_data.loc[tuning_valid_mask, ["weekly_sales"]].copy()

    check["prediction"] = np.maximum(
        0,
        candidate.predict(X_tuning_valid),
    )

    metrics = evaluate_forecasts(
        check,
        {"Optuna": "prediction"},
    ).loc["Optuna"]

    trial.set_user_attr("trees", int(candidate.tree_count_))
    trial.set_user_attr("WAPE", float(metrics["WAPE"]))
    trial.set_user_attr("Bias", float(metrics["Bias"]))
    trial.set_user_attr("seconds", perf_counter() - started)

    return float(metrics["MAE"])


TOTAL_TRIALS = 10
remaining_trials = max(0, TOTAL_TRIALS - len(study.trials))

print(f"Новых обучений: {remaining_trials}")

if remaining_trials:
    study.optimize(
        objective,
        n_trials=remaining_trials,
        n_jobs=1,
        gc_after_trial=True,
    )

Новых обучений: 6
0:	learn: 9.8468276	test: 7.9301017	best: 7.9301017 (0)	total: 201ms	remaining: 16m 43s


[I 2026-10-20 12:44:09,938] Trial 4 finished with value: 3.350717635510372 and parameters: {'depth': 7, 'learning_rate': 0.05684076562897628, 'l2_leaf_reg': 19.28670650737206}. Best is trial 0 with value: 3.2829232187741173.


Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.351435632
bestIteration = 152

Shrink model to first 153 iterations.
0:	learn: 9.7619414	test: 7.8463926	best: 7.8463926 (0)	total: 119ms	remaining: 9m 56s
500:	learn: 3.9929474	test: 3.3632032	best: 3.3632032 (498)	total: 40.8s	remaining: 6m 6s
1000:	learn: 3.9112618	test: 3.3331043	best: 3.3331043 (1000)	total: 1m 25s	remaining: 5m 39s
1500:	learn: 3.8563690	test: 3.3160595	best: 3.3154874 (1477)	total: 2m 11s	remaining: 5m 5s
2000:	learn: 3.8263495	test: 3.3055301	best: 3.3055239 (1999)	total: 2m 57s	remaining: 4m 25s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.303573707
bestIteration = 2100

Shrink model to first 2101 iterations.


[I 2026-10-20 12:47:26,786] Trial 5 finished with value: 3.3011144324147 and parameters: {'depth': 5, 'learning_rate': 0.08276715484830396, 'l2_leaf_reg': 9.625053699813357}. Best is trial 0 with value: 3.2829232187741173.


0:	learn: 9.9963390	test: 8.0687910	best: 8.0687910 (0)	total: 97.6ms	remaining: 8m 8s
500:	learn: 4.0880447	test: 3.3681440	best: 3.3681440 (500)	total: 40.5s	remaining: 6m 3s


...

[I 2026-10-20 12:49:00,615] Trial 7 finished with value: 3.358952595084185 and parameters: {'depth': 5, 'learning_rate': 0.03076066766601349, 'l2_leaf_reg': 1.2096182373800828}. Best is trial 0 with value: 3.2829232187741173.


Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.359678885
bestIteration = 324

Shrink model to first 325 iterations.
0:	learn: 9.9701258	test: 8.0458898	best: 8.0458898 (0)	total: 184ms	remaining: 15m 18s
500:	learn: 3.9774182	test: 3.3414077	best: 3.3414077 (500)	total: 1m	remaining: 9m 4s
1000:	learn: 3.8399059	test: 3.2968067	best: 3.2967586 (998)	total: 2m 8s	remaining: 8m 32s
1500:	learn: 3.7840025	test: 3.2818962	best: 3.2818962 (1500)	total: 3m 15s	remaining: 7m 36s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.278539846
bestIteration = 1724

Shrink model to first 1725 iterations.


[I 2026-10-20 12:53:02,151] Trial 8 finished with value: 3.277228657330277 and parameters: {'depth': 8, 'learning_rate': 0.030447981257860905, 'l2_leaf_reg': 6.287286867969111}. Best is trial 8 with value: 3.277228657330277.


0:	learn: 9.7284377	test: 7.8209280	best: 7.8209280 (0)	total: 187ms	remaining: 15m 32s
500:	learn: 3.8315512	test: 3.3057425	best: 3.3056431 (499)	total: 1m 8s	remaining: 10m 12s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.292797101
bestIteration = 703

Shrink model to first 704 iterations.


[I 2026-10-20 12:54:54,038] Trial 9 finished with value: 3.290575993529976 and parameters: {'depth': 8, 'learning_rate': 0.07633573707911982, 'l2_leaf_reg': 7.340952323327929}. Best is trial 8 with value: 3.277228657330277.


### 8.4. Победитель подбора

Лучший вариант получил глубину **8**, `learning_rate≈0.030448` и `l2_leaf_reg≈6.287287`. На периоде подбора выбрано 1 725 деревьев, MAE — 3.2772. Выигрыш относительно контроля — 0.42%.

Это результат выбора параметров на одном периоде. Дальше проверяем выбранный вариант на тех же четырёх временных окнах.

In [32]:
optuna_results = (
    pd.DataFrame(
        [
            {
                "trial": trial.number,
                "variant": trial.user_attrs.get("manual_variant", "Optuna"),
                **trial.params,
                "trees": trial.user_attrs["trees"],
                "MAE": trial.value,
                "WAPE": trial.user_attrs["WAPE"],
                "Bias": trial.user_attrs["Bias"],
                "seconds": trial.user_attrs["seconds"],
            }
            for trial in study.trials
            if trial.state == TrialState.COMPLETE
        ]
    )
    .set_index("trial")
    .sort_values("MAE")
)

control_mae = float(tuning_table.loc["Контроль", "MAE"])
optuna_results["gain"] = 1 - optuna_results["MAE"] / control_mae

display(
    optuna_results.style.format(
        {
            "learning_rate": "{:.4f}",
            "l2_leaf_reg": "{:.3f}",
            "MAE": "{:.4f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
            "seconds": "{:.0f}",
            "gain": "{:+.2%}",
        }
    )
)

best_optuna_params = {
    **base_params,
    **study.best_params,
    "iterations": 5000,
    "allow_writing_files": False,
}

best_optuna_trees = study.best_trial.user_attrs["trees"]

print("Лучшие параметры:", study.best_params)
print("Деревьев на периоде подбора:", best_optuna_trees)

,variant,depth,learning_rate,l2_leaf_reg,trees,MAE,WAPE,Bias,seconds,gain
trial,,,,,,,,,,
8,Optuna,8,0.0304,6.287,1725,3.2772,34.84%,-5.88%,242,+0.42%
0,Регуляризация 10,6,0.0500,10.000,2290,3.2829,34.90%,-6.02%,259,+0.25%
9,Optuna,8,0.0763,7.341,704,3.2906,34.98%,-6.01%,112,+0.01%
1,Контроль,6,0.0500,3.000,1918,3.2910,34.98%,-6.00%,199,+0.00%
5,Optuna,5,0.0828,9.625,2101,3.3011,35.09%,-6.03%,197,-0.31%
2,Глубина 8,8,0.0500,3.000,161,3.3464,35.57%,-5.93%,35,-1.68%
4,Optuna,7,0.0568,19.287,153,3.3507,35.62%,-5.97%,30,-1.81%
6,Optuna,4,0.0315,19.456,598,3.3581,35.69%,-5.82%,54,-2.04%
7,Optuna,5,0.0308,1.210,325,3.3590,35.70%,-5.91%,40,-2.06%


Лучшие параметры: {'depth': 8, 'learning_rate': 0.030447981257860905, 'l2_leaf_reg': 6.287286867969111}
Деревьев на периоде подбора: 1725


<a id="tuned-validation"></a>

## 9. Проверка победителя Optuna

Повторяем четыре окна с выбранными параметрами. Для первых трёх заново выбираем число деревьев на предшествующих неделях. Для последнего используем уже найденные 1 725 деревьев, поскольку период подбора совпадает.

Всего выполняется семь обучений. Результаты контроля берём из раздела 7.

Эти окна относятся к разработке модели. Более того, параметры выбраны на позднем периоде и применены к более ранним окнам, поэтому это ретроспективная проверка устойчивости, а не четыре независимых финальных теста.

In [33]:
tuned_cv_params = {
    **best_optuna_params,
    "iterations": 5000,
    "allow_writing_files": False,
}

tuned_cv_parts = []
tuned_cv_rows = []

dates = cv_data["week_end"]

for window in cv_windows:
    fold = window["fold"]
    started = perf_counter()

    inner_train_mask = cv_eligible & dates.lt(window["tune_start"])

    inner_stop_mask = cv_eligible & dates.ge(window["tune_start"]) & dates.lt(window["score_start"])

    past_mask = cv_eligible & dates.lt(window["score_start"])

    score_mask = cv_eligible & dates.ge(window["score_start"]) & dates.le(window["score_end"])

    print(
        f"\nОкно {fold}/{len(cv_windows)}: "
        f"{window['score_start']:%Y-%m-%d} — "
        f"{window['score_end']:%Y-%m-%d}"
    )

    if window == tuning_window:
        selected_trees = int(best_optuna_trees)
        print(f"Используем выбранные Optuna {selected_trees} деревьев.")

    else:
        tuned_selector = CatBoostRegressor(**tuned_cv_params)

        tuned_selector.fit(
            cv_X.loc[inner_train_mask],
            cv_y.loc[inner_train_mask],
            cat_features=cv_categorical,
            eval_set=(
                cv_X.loc[inner_stop_mask],
                cv_y.loc[inner_stop_mask],
            ),
            early_stopping_rounds=100,
            use_best_model=True,
            verbose=500,
        )

        selected_trees = int(tuned_selector.tree_count_)

    print(f"Обучаем на всём прошлом: {selected_trees} деревьев.")

    tuned_cv_model = CatBoostRegressor(
        **{
            **tuned_cv_params,
            "iterations": selected_trees,
        }
    )

    tuned_cv_model.fit(
        cv_X.loc[past_mask],
        cv_y.loc[past_mask],
        cat_features=cv_categorical,
        use_best_model=False,
        verbose=500,
    )

    fold_predictions = cv_data.loc[
        score_mask,
        ["store_id", "item_id", "week_end"],
    ].copy()

    fold_predictions["prediction_tuned"] = np.maximum(
        0,
        tuned_cv_model.predict(cv_X.loc[score_mask]),
    )

    fold_predictions["fold"] = fold
    tuned_cv_parts.append(fold_predictions)

    tuned_cv_rows.append(
        {
            "fold": fold,
            "score_start": window["score_start"],
            "score_end": window["score_end"],
            "trees": selected_trees,
            "seconds": perf_counter() - started,
        }
    )

    print(f"Окно готово за {perf_counter() - started:.0f} секунд.")


Окно 1/4: 2015-05-10 — 2015-06-28
0:	learn: 10.2997734	test: 8.5723963	best: 8.5723963 (0)	total: 182ms	remaining: 15m 10s
500:	learn: 4.0525660	test: 3.6584647	best: 3.6584647 (500)	total: 47.8s	remaining: 7m 9s
[Промежуточный журнал обучения сокращён; метрики и графики сохранены.]
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.59376902
bestIteration = 2395

Shrink model to first 2396 iterations.
Обучаем на всём прошлом: 2396 деревьев.
Окно готово за 550 секунд.

Окно 2/4: 2015-08-09 — 2015-09-27
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.585417911
bestIteration = 1088

Shrink model to first 1089 iterations.
Обучаем на всём прошлом: 1089 деревьев.
Окно готово за 269 секунд.

Окно 3/4: 2015-11-08 — 2015-12-27
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.483659733
bestIteration = 1594

Shrink model to first 1595 iterations.
Обучаем на всём прошлом: 1595 деревьев.
Окно готово за 413 секунд.

Окно 4/4: 2016-02-07 — 2

Сопоставляем прогнозы по магазину, товару, неделе и номеру окна. Проверка объединения гарантирует, что контроль и новый вариант оцениваются на одинаковых наблюдениях.

In [34]:
tuned_cv_predictions = pd.concat(
    tuned_cv_parts,
    ignore_index=True,
)

comparison_keys = ["fold", "store_id", "item_id", "week_end"]

cv_comparison = cv_predictions.merge(
    tuned_cv_predictions,
    on=comparison_keys,
    how="left",
    validate="one_to_one",
)

assert len(cv_comparison) == len(tuned_cv_predictions)
assert cv_comparison["prediction_tuned"].notna().all()

cv_comparison["ae_control"] = (cv_comparison["prediction_cv"] - cv_comparison["weekly_sales"]).abs()

cv_comparison["ae_tuned"] = (
    cv_comparison["prediction_tuned"] - cv_comparison["weekly_sales"]
).abs()

fold_mae = (
    cv_comparison.groupby("fold")[["ae_control", "ae_tuned"]]
    .mean()
    .rename(
        columns={
            "ae_control": "MAE_control",
            "ae_tuned": "MAE_Optuna",
        }
    )
)

tuned_cv_summary = pd.DataFrame(tuned_cv_rows).set_index("fold").join(fold_mae)

tuned_cv_summary["gain"] = 1 - tuned_cv_summary["MAE_Optuna"] / tuned_cv_summary["MAE_control"]

display(
    tuned_cv_summary.style.format(
        {
            "score_start": "{:%Y-%m-%d}",
            "score_end": "{:%Y-%m-%d}",
            "seconds": "{:.0f}",
            "MAE_control": "{:.4f}",
            "MAE_Optuna": "{:.4f}",
            "gain": "{:+.2%}",
        }
    )
)

tuned_cv_overall = evaluate_forecasts(
    cv_comparison,
    {
        "Среднее за 4 недели": BASELINE_METHODS["Среднее за 4 недели"],
        "CatBoost — контроль": "prediction_cv",
        "CatBoost — Optuna": "prediction_tuned",
    },
)

display(
    tuned_cv_overall.style.format(
        {
            "MAE": "{:.4f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

wins = int(tuned_cv_summary["gain"].gt(0).sum())
print(f"Optuna лучше контроля в {wins} из {len(cv_windows)} окон.")

,score_start,score_end,trees,seconds,MAE_control,MAE_Optuna,gain
fold,,,,,,,
1,2015-05-10,2015-06-28,2396,550,3.5206,3.4956,+0.71%
2,2015-08-09,2015-09-27,1089,269,3.4775,3.4740,+0.10%
3,2015-11-08,2015-12-27,1595,413,3.3185,3.3001,+0.56%
4,2016-02-07,2016-03-27,1725,256,3.3228,3.3181,+0.14%


,MAE,WAPE,Bias
Метод,,,
Среднее за 4 недели,3.7391,36.92%,+0.33%
CatBoost — контроль,3.4097,33.67%,-4.91%
CatBoost — Optuna,3.3968,33.54%,-4.43%


Optuna лучше контроля в 4 из 4 окон.


После Optuna общий MAE снизился с **3.4097 до 3.3968** — ещё на 0.38%. Улучшение есть во всех четырёх окнах, но само по себе небольшое. WAPE — 33.54%, Bias — −4.43%.

Для финальной проверки выбираем этот вариант. Настройки фиксируем до просмотра ошибок на новых четырёх неделях.

<a id="final-test"></a>

## 10. Финальный период: 25 апреля — 22 мая 2016

### 10.1. Данные и признаки

Фиксируем параметры, список признаков и категориальные столбцы. Из уже загруженного архива читаем `sales_train_evaluation.csv` только для CA_1: в нём есть ещё 28 дней продаж.

Проверяем совпадение прежней истории, заново собираем недели и **те же 27 признаков**. Календари событий и история цен уже доступны. Все лаги и скользящие продажи используют только завершившиеся недели.

Проверочные недели заканчиваются **1, 8, 15 и 22 мая**. Модель оценивает четыре последовательных прогноза на одну неделю вперёд; перед каждым признаки получают факты предыдущей завершившейся недели.

In [35]:
from zipfile import ZipFile

FINAL_TEST_START = pd.Timestamp("2016-04-25")
FINAL_TEST_END = pd.Timestamp("2016-05-22")

frozen_params = {
    **best_optuna_params,
    "iterations": 5000,
    "allow_writing_files": False,
}
frozen_features = list(features_all)
frozen_categorical = list(cv_categorical)

evaluation_parts = []

with ZipFile(DATA_RAW / "m5-forecasting-accuracy.zip") as archive:
    members = [
        name for name in archive.namelist() if Path(name).name == "sales_train_evaluation.csv"
    ]
    assert len(members) == 1
    member = members[0]

    with archive.open(member) as source:
        evaluation_columns = pd.read_csv(source, nrows=0).columns

    evaluation_dtypes = {
        column: "int32" if column.startswith("d_") else "string" for column in evaluation_columns
    }

    with archive.open(member) as source:
        for chunk in pd.read_csv(
            source,
            dtype=evaluation_dtypes,
            chunksize=5000,
        ):
            selected = chunk.loc[chunk["store_id"].eq(STORE_ID)]
            if not selected.empty:
                evaluation_parts.append(selected.copy())

sales_extended = pd.concat(evaluation_parts, ignore_index=True)

assert set(sales_extended["item_id"]) == set(sales["item_id"])

historical_days = [column for column in sales.columns if column.startswith("d_")]
series_keys = ["store_id", "item_id"]

pd.testing.assert_frame_equal(
    sales.set_index(series_keys)[historical_days].sort_index(),
    sales_extended.set_index(series_keys)[historical_days].sort_index(),
    check_dtype=False,
)

weekly_extended = build_weekly_table(
    calendar,
    sales_extended,
    prices,
)

final_data = add_weekly_features(weekly_extended)

sales_groups = final_data.groupby(
    series_keys,
    observed=True,
    sort=False,
)["weekly_sales"]

for window in (2, 13):
    final_data[f"mean_{window}w"] = sales_groups.transform(
        lambda series: series.shift(1).rolling(window).mean()
    )

final_data["std_8w"] = sales_groups.transform(lambda series: series.shift(1).rolling(8).std())
final_data["trend_2w_8w"] = final_data["mean_2w"] - final_data["mean_8w"]

final_data["zero_share_8w"] = sales_groups.transform(zero_share)
final_data["zero_run"] = sales_groups.transform(zero_run)

final_data["mean_nonzero_8w"] = sales_groups.transform(
    lambda series: (series.shift(1).replace(0, np.nan).rolling(8, min_periods=1).mean())
)

final_data = final_data.join(
    calendar_weekly[calendar_features],
    on="week_end",
    validate="many_to_one",
)

final_data = final_data.join(
    price_lookup[["price_quote"]],
    on=[*series_keys, "week_end"],
    validate="many_to_one",
)

final_data["price_missing_last_period"] = final_data["price_quote"].isna().astype("int8")

final_data["price_last"] = final_data.groupby(
    series_keys,
    observed=True,
    sort=False,
)["price_quote"].ffill()

price_groups = final_data.groupby(
    series_keys,
    observed=True,
    sort=False,
)["price_last"]

final_data["price_change_1w"] = final_data["price_last"] / price_groups.shift(1) - 1

mean_price_4w = price_groups.transform(lambda series: series.rolling(4, min_periods=1).mean())
final_data["price_relative_4w"] = final_data["price_last"] / mean_price_4w - 1

final_X = final_data[frozen_features].copy()

for column in frozen_categorical:
    final_X[column] = final_X[column].astype(str).astype(object)

final_y = final_data["weekly_sales"]
final_dates = final_data["week_end"]

final_eligible = final_data["known_at_origin"] & final_data[NUMERIC_FEATURES].notna().all(axis=1)

observed_test_weeks = (
    final_dates.loc[final_dates.between(FINAL_TEST_START, FINAL_TEST_END)]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

assert observed_test_weeks == list(
    pd.date_range(
        FINAL_TEST_START,
        FINAL_TEST_END,
        freq="W-SUN",
    )
)

print(f"Товаров: {sales_extended['item_id'].nunique():,}")
print(f"Признаков: {len(frozen_features)}")
print("Проверочные недели:", [date.strftime("%Y-%m-%d") for date in observed_test_weeks])

Товаров: 3,049
Признаков: 27
Проверочные недели: ['2016-05-01', '2016-05-08', '2016-05-15', '2016-05-22']


### 10.2. Два обучения и оценка

Сначала выбираем число деревьев на последних восьми неделях до финального периода. Затем обучаем модель на всей подходящей истории до 25 апреля с фиксированным числом деревьев. Финальные четыре недели в ранней остановке не участвуют.

В выполненном запуске выбрано **1 952 дерева**. На финальном периоде сравниваем модель с пятью простыми методами и сохраняем:

- `models/catboost_ca1_final.cbm` — модель;
- `results/ca1_scaling/final_test/config.json` — параметры и признаки;
- `results/ca1_scaling/final_test/predictions.csv` — факты и прогнозы;
- `results/ca1_scaling/final_test/metrics.csv` — общие ошибки.

In [36]:
past_mask = final_eligible & final_dates.lt(FINAL_TEST_START)

past_weeks = final_dates.loc[final_dates.lt(FINAL_TEST_START)].drop_duplicates().sort_values()

stop_start = past_weeks.iloc[-8]

selector_train_mask = final_eligible & final_dates.lt(stop_start)

selector_stop_mask = past_mask & final_dates.ge(stop_start)

final_test_mask = final_eligible & final_dates.between(FINAL_TEST_START, FINAL_TEST_END)

print("Обучение 1/2: выбираем число деревьев.")

final_selector = CatBoostRegressor(**frozen_params)

final_selector.fit(
    final_X.loc[selector_train_mask],
    final_y.loc[selector_train_mask],
    cat_features=frozen_categorical,
    eval_set=(
        final_X.loc[selector_stop_mask],
        final_y.loc[selector_stop_mask],
    ),
    early_stopping_rounds=100,
    use_best_model=True,
    verbose=500,
)

final_model_params = {
    **frozen_params,
    "iterations": int(final_selector.tree_count_),
}

print(f"\nОбучение 2/2: вся история до теста, " f"{final_model_params['iterations']} деревьев.")

final_model = CatBoostRegressor(**final_model_params)

final_model.fit(
    final_X.loc[past_mask],
    final_y.loc[past_mask],
    cat_features=frozen_categorical,
    use_best_model=False,
    verbose=500,
)

final_predictions = final_data.loc[
    final_test_mask,
    [
        "store_id",
        "item_id",
        "cat_id",
        "dept_id",
        "week_end",
        "weekly_sales",
        *BASELINE_METHODS.values(),
        "mean_8w",
        "mean_13w",
    ],
].copy()

final_predictions["prediction_final"] = np.maximum(
    0,
    final_model.predict(final_X.loc[final_test_mask]),
)

final_methods = {
    **BASELINE_METHODS,
    "Среднее за 8 недель": "mean_8w",
    "Среднее за 13 недель": "mean_13w",
    "CatBoost — Optuna": "prediction_final",
}

final_metrics = evaluate_forecasts(
    final_predictions,
    final_methods,
)

display(
    final_metrics.sort_values("MAE").style.format(
        {
            "MAE": "{:.4f}",
            "WAPE": "{:.2%}",
            "Bias": "{:+.2%}",
        }
    )
)

final_baseline_mae = final_metrics.loc["Среднее за 4 недели", "MAE"]
final_model_mae = final_metrics.loc["CatBoost — Optuna", "MAE"]

print(
    "Снижение MAE относительно среднего за 4 недели: "
    f"{1 - final_model_mae / final_baseline_mae:+.2%}"
)

final_dir = RESULTS / "ca1_scaling" / "final_test"
final_dir.mkdir(parents=True, exist_ok=True)

final_predictions.to_csv(
    final_dir / "predictions.csv",
    index=False,
)
final_metrics.to_csv(final_dir / "metrics.csv")

(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)
final_model.save_model(str(PROJECT_ROOT / "models" / "catboost_ca1_final.cbm"))

final_config = {
    "params": final_model_params,
    "features": frozen_features,
    "categorical_features": frozen_categorical,
    "test_start": str(FINAL_TEST_START.date()),
    "test_end": str(FINAL_TEST_END.date()),
    "forecast_protocol": "rolling_one_week_fixed_model",
}

(final_dir / "config.json").write_text(
    json.dumps(final_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

Обучение 1/2: выбираем число деревьев.
0:	learn: 9.8845769	test: 8.5561467	best: 8.5561467 (0)	total: 237ms	remaining: 19m 43s
500:	learn: 3.9407193	test: 3.3899861	best: 3.3895814 (438)	total: 1m 11s	remaining: 10m 38s
1000:	learn: 3.8352866	test: 3.3704527	best: 3.3704527 (1000)	total: 2m 16s	remaining: 9m 5s
1500:	learn: 3.7791715	test: 3.3600892	best: 3.3600892 (1500)	total: 3m 24s	remaining: 7m 57s
2000:	learn: 3.7290321	test: 3.3550945	best: 3.3548674 (1951)	total: 4m 38s	remaining: 6m 58s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.354867406
bestIteration = 1951

Shrink model to first 1952 iterations.

Обучение 2/2: вся история до теста, 1952 деревьев.
0:	learn: 9.8216477	total: 197ms	remaining: 6m 24s
500:	learn: 3.9770854	total: 1m 2s	remaining: 2m 59s
1000:	learn: 3.8208493	total: 2m 14s	remaining: 2m 8s
1500:	learn: 3.7643991	total: 3m 29s	remaining: 1m 2s
1951:	learn: 3.7261618	total: 4m 35s	remaining: 0us


,MAE,WAPE,Bias
Метод,,,
CatBoost — Optuna,3.5512,32.31%,-5.84%
Среднее за 4 недели,3.8097,34.67%,-3.65%
Медиана за 4 недели,3.8931,35.42%,-5.60%
Среднее за 8 недель,4.0317,36.69%,-4.55%
Среднее за 13 недель,4.2182,38.38%,-5.94%
Предыдущая неделя,4.2360,38.54%,-1.11%


Снижение MAE относительно среднего за 4 недели: +6.79%


989

На финальном периоде CatBoost получил **MAE 3.5512, WAPE 32.31%, Bias −5.84%**. У среднего за четыре недели MAE 3.8097: снижение ошибки составляет **6.79%**.

По суммарному объёму среднее оказалось ближе к факту: его Bias −3.65%. Поэтому у выбранной модели есть два разных свойства: меньшая абсолютная ошибка и более сильное занижение общего объёма продаж.

<a id="errors"></a>

## 11. Ошибки по неделям и категориям

Используем готовые прогнозы, новые модели не обучаем. Считаем те же метрики отдельно по неделям и категориям. В столбце `gain` положительное значение означает снижение MAE относительно среднего за четыре недели.

Суммы продаж нужны для контекста: категории с разными объёмами могут иметь разный MAE даже при сходном качестве. Разрезы сохраняем рядом с общими результатами.

In [37]:
def error_report(group_column):
    rows = []

    methods = {
        "Бейзлайн": "prediction_mean_4w",
        "CatBoost": "prediction_final",
    }

    for group_name, part in final_predictions.groupby(
        group_column,
        observed=True,
        sort=True,
    ):
        metrics = evaluate_forecasts(part, methods)

        baseline_mae = metrics.loc["Бейзлайн", "MAE"]
        model_mae = metrics.loc["CatBoost", "MAE"]

        rows.append(
            {
                group_column: group_name,
                "actual_sales": int(part["weekly_sales"].sum()),
                "MAE_mean_4w": baseline_mae,
                "MAE_CatBoost": model_mae,
                "gain": (1 - model_mae / baseline_mae if baseline_mae > 0 else np.nan),
                "WAPE_CatBoost": metrics.loc["CatBoost", "WAPE"],
                "Bias_CatBoost": metrics.loc["CatBoost", "Bias"],
            }
        )

    return pd.DataFrame(rows).set_index(group_column)


weekly_errors = error_report("week_end")
category_errors = error_report("cat_id")

report_format = {
    "actual_sales": "{:,.0f}",
    "MAE_mean_4w": "{:.4f}",
    "MAE_CatBoost": "{:.4f}",
    "gain": "{:+.2%}",
    "WAPE_CatBoost": "{:.2%}",
    "Bias_CatBoost": "{:+.2%}",
}

print("Ошибки по неделям")
display(weekly_errors.style.format(report_format).format_index("{:%Y-%m-%d}", axis=0))

print("Ошибки по категориям")
display(category_errors.style.format(report_format))

weekly_errors.to_csv(final_dir / "metrics_by_week.csv")
category_errors.to_csv(final_dir / "metrics_by_category.csv")

Ошибки по неделям


,actual_sales,MAE_mean_4w,MAE_CatBoost,gain,WAPE_CatBoost,Bias_CatBoost
week_end,,,,,,
2016-05-01,"32,376",3.7910,3.4178,+9.84%,32.19%,-6.26%
2016-05-08,"33,625",4.0107,3.6678,+8.55%,33.26%,-2.05%
2016-05-15,"34,897",3.8022,3.5570,+6.45%,31.08%,-9.20%
2016-05-22,"33,134",3.6350,3.5621,+2.00%,32.78%,-5.73%


Ошибки по категориям


,actual_sales,MAE_mean_4w,MAE_CatBoost,gain,WAPE_CatBoost,Bias_CatBoost
cat_id,,,,,,
FOODS,"90,833",5.0737,4.6948,+7.47%,29.71%,-5.83%
HOBBIES,"15,405",2.9284,2.8102,+4.04%,41.23%,-9.62%
HOUSEHOLD,"27,794",2.5505,2.3815,+6.63%,35.88%,-3.77%


<a id="conclusions"></a>

## 12. Вывод

CatBoost с глубиной 8 и 1 952 деревьями уменьшил MAE на **6.79%** относительно
среднего четырёх недель. Выигрыш есть во всех четырёх проверочных неделях и категориях:
FOODS **7.47%**, HOUSEHOLD **6.63%**, HOBBIES **4.04%**.

Общий Bias — **−5.84%**. У HOBBIES наибольшие WAPE (**41.23%**) и недопрогноз
(**−9.62%**). Четыре недели одного магазина ограничивают выводы о переносимости.

Метрики сохранены в [results/ca1_scaling/final_test](../results/ca1_scaling/final_test).
Дневной прогноз всех магазинов и WRMSSE разобраны в [ноутбуке 03](03_kaggle_wrmsse.ipynb).